<a href="https://colab.research.google.com/github/rajkumarsingh19/LLM_usecase/blob/main/RAG_Pipeline_with_LLM_Embeddings_and_Scikit_learn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q sentence-transformers transformers

import os
import glob
import urllib.request
import zipfile
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from sentence_transformers import SentenceTransformer
from transformers import pipeline

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [ ]:
# Downloading and extracting the toy dataset, containing 9 documents
url = "https://github.com/gakudo-ai/open-datasets/blob/main/asia_documents.zip?raw=true"
zip_path = "asia_documents.zip"
extract_dir = "asia_documents"

urllib.request.urlretrieve(url, zip_path)
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

# Reading all extracted text files
documents = []
for filepath in glob.glob(f"{extract_dir}/**/*.txt", recursive=True):
    with open(filepath, 'r', encoding='utf-8') as file:
        documents.append(file.read())

print(f"Loaded {len(documents)} documents.")
# If the zip contained a CSV instead, you'd use pd.read_csv() here.

Loaded 9 documents.


In [ ]:
# Initializing the embedding model (runs locally on Colab's free CPU/GPU)
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Generating embeddings for all text chunks
print("Generating embeddings... (this is fast on Colab!)")
chunk_embeddings = embedding_model.encode(all_chunks, show_progress_bar=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Generating embeddings... (this is fast on Colab!)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
class SklearnRetriever:
    def __init__(self, n_neighbors=3):
        # Using 'cosine' metric because LLM embeddings measure semantic similarity based on angles between vectors
        self.retriever = NearestNeighbors(n_neighbors=n_neighbors, metric='cosine', algorithm='brute')
        self.chunks = []

    def fit(self, chunks, embeddings):
        """Indexes the documents."""
        self.chunks = chunks
        self.retriever.fit(embeddings)
        print("Vector index built successfully!")

    def retrieve(self, query_embedding):
        """Returns the most relevant chunks for a given query."""
        distances, indices = self.retriever.kneighbors([query_embedding])

        # Fetching the actual text strings using the retrieved indices
        results = [self.chunks[idx] for idx in indices[0]]
        return results

# Instantiating and fitting our custom retriever
vector_db = SklearnRetriever(n_neighbors=3)
vector_db.fit(all_chunks, chunk_embeddings)

Vector index built successfully!


In [ ]:
# Loading a lightweight causal LLM built for the "text-generation" pipeline
llm = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", device_map="auto")

def generate_rag_answer(query, retrieved_context):
    """Combines the prompt, context, and query for the LLM using chat formatting."""
    context_str = " --- ".join(retrieved_context)

    messages = [
        {"role": "system", "content": "You are a helpful assistant. Answer the user's question using ONLY the provided context. If the answer is not in the context, say 'I cannot find the answer in the provided documents.'"},
        {"role": "user", "content": f"Context:\n{context_str}\n\nQuestion: {query}"}
    ]

    # Apply the model's chat template
    prompt = llm.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # Generating the response
    response = llm(prompt, max_new_tokens=150, do_sample=False, temperature=0.1)

    # Extracting only the newly generated text past the prompt
    generated_text = response[0]['generated_text']
    answer = generated_text[len(prompt):].strip()
    return answer

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [ ]:
def ask_rag_pipeline(question):
    print(f"Question: {question}")

    # 1. Embedding the query
    query_vector = embedding_model.encode(question)

    # 2. Retrieving context via Scikit-Learn
    relevant_chunks = vector_db.retrieve(query_vector)

    # 3. Generating Answer
    answer = generate_rag_answer(question, relevant_chunks)

    print(f"\nRAG Answer: {answer}")
    print("\n--- Sources used ---")
    for i, chunk in enumerate(relevant_chunks):
        print(f"{i+1}. {chunk[:100]}...") # Truncated for readability

# Testing it out
test_question = "Which Asian countries are best known for islands and beaches?"
ask_rag_pipeline(test_question)

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Question: Which Asian countries are best known for islands and beaches?


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



RAG Answer: The Asian countries that are best known for islands and beaches include:

1. Philippines - Known for its over 7,000 islands and vibrant festivals.
2. Thailand - Famous for its tropical beaches and bustling street food culture.

These two countries have a significant number of islands and offer both beautiful natural scenery and diverse beach experiences.

--- Sources used ---
1. mountainous landscapes and cultural cities like Chiang Mai, while the south offers world-renowned is...
2. The Philippines is an archipelago of over 7,000 islands in Southeast Asia, known for its stunning be...
3. Thailand is a Southeast Asian country famous for its tropical beaches, ornate temples, and bustling ...


In [ ]:
def ask_rag_pipeline(question):
    print(f"Question: {question}")

    # 1. Embedding the query
    query_vector = embedding_model.encode(question)

    # 2. Retrieving context via Scikit-Learn
    relevant_chunks = vector_db.retrieve(query_vector)

    # 3. Generating Answer
    answer = generate_rag_answer(question, relevant_chunks)

    print(f"\nRAG Answer: {answer}")
    print("\n--- Sources used ---")
    for i, chunk in enumerate(relevant_chunks):
        print(f"{i+1}. {chunk[:100]}...") # Truncated for readability

# Testing it out
test_question = "Which Asian countries are best known for islands and beaches?"
ask_rag_pipeline(test_question)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: Which Asian countries are best known for islands and beaches?

RAG Answer: The Asian countries that are best known for islands and beaches include:

1. Philippines - Known for its over 7,000 islands and vibrant festivals.
2. Thailand - Famous for its tropical beaches and bustling street food culture.

These two countries have a significant number of islands and offer both beautiful natural scenery and diverse beach experiences.

--- Sources used ---
1. mountainous landscapes and cultural cities like Chiang Mai, while the south offers world-renowned is...
2. The Philippines is an archipelago of over 7,000 islands in Southeast Asia, known for its stunning be...
3. Thailand is a Southeast Asian country famous for its tropical beaches, ornate temples, and bustling ...
